In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

## load from kaggle

In [3]:
# first download to see what files exist
dataset_path = kagglehub.dataset_download("rivalcults/youtube-scam-phone-call-transcripts")
print(f"dataset downloaded to: {dataset_path}")

import os
files = os.listdir(dataset_path)
print(f"files: {files}")

100%|██████████| 146k/146k [00:00<00:00, 3.72MB/s]

Extracting files...
dataset downloaded to: /Users/jnkim/.cache/kagglehub/datasets/rivalcults/youtube-scam-phone-call-transcripts/versions/2
files: ['FullTranscriptData.xlsx', 'FullTranscriptData.csv']


In [4]:
# load the csv
df = pd.read_csv(os.path.join(dataset_path, "FullTranscriptData.csv"))
print(f"shape: {df.shape}")
df.head()

shape: (243, 4)


,ID,Source,Content,Char_Len
0,0,https://www.youtube.com/watch?v=UG-KjCSt14k&ab...,hi yeah I've got an email here about a laptop ...,224
1,1,https://www.patreon.com/JimBrowning/posts?filt...,Hello This is British Telecom technical depart...,398
2,2,https://www.youtube.com/watch?v=P6dhteJIY48&ab...,hello this call is from Interpol the purpose o...,186
3,3,https://www.youtube.com/watch?v=HlTpousFPsM&ab...,hi this is from Amazon customer service we hav...,361
4,4,https://www.youtube.com/watch?v=8sT121uUFOo&ab...,this is calling with the vehicle service depar...,411


In [6]:
# this dataset only has scam calls, no legitimate calls
# so we can't do classification, but we can analyze scam characteristics
print("columns:", df.columns.tolist())
print(f"\ntotal scam transcripts: {len(df)}")

columns: ['ID', 'Source', 'Content', 'Char_Len']

total scam transcripts: 243


## extract features

In [7]:
def count_urgency_words(text):
    urgency_words = ['immediately', 'urgent', 'now', 'today', 'expires', 'hurry', 
                     'asap', 'quickly', 'deadline', 'limited time']
    text_lower = str(text).lower()
    return sum(1 for word in urgency_words if word in text_lower)

def count_verification_requests(text):
    verification_patterns = [
        r'verify\s+your', r'confirm\s+your', r'provide\s+your',
        r'enter\s+your', r'social\s+security', r'account\s+number',
        r'password', r'credit\s+card', r'bank\s+account'
    ]
    return sum(1 for pattern in verification_patterns if re.search(pattern, str(text).lower()))

In [9]:
# text is in 'Content' column
text_col = 'Content'
print(f"using column: {text_col}")

df['urgency_score'] = df[text_col].apply(count_urgency_words)
df['verification_requests'] = df[text_col].apply(count_verification_requests)

print(f"\naverage urgency score: {df['urgency_score'].mean():.2f}")
print(f"average verification requests: {df['verification_requests'].mean():.2f}")

# top examples with high urgency
print(f"\ntranscripts with urgency > 0: {(df['urgency_score'] > 0).sum()}")
print(f"transcripts with verification requests > 0: {(df['verification_requests'] > 0).sum()}")

using column: Content

average urgency score: 1.03
average verification requests: 0.07

transcripts with urgency > 0: 189
transcripts with verification requests > 0: 17


## baseline model

In [10]:
# analyze word frequency in scam calls
from sklearn.feature_extraction.text import CountVectorizer

# get most common words
vectorizer = CountVectorizer(stop_words='english', max_features=20, ngram_range=(1, 2))
word_counts = vectorizer.fit_transform(df['Content'])
word_freq = word_counts.sum(axis=0).A1
word_names = vectorizer.get_feature_names_out()

# sort by frequency
top_indices = np.argsort(word_freq)[::-1]
print("most common words/phrases in scam calls:")
for idx in top_indices[:15]:
    print(f"  {word_names[idx]}: {word_freq[idx]}")

most common words/phrases in scam calls:
  okay: 533
  computer: 350
  uh: 335
  yes: 315
  right: 305
  yeah: 300
  help: 298
  just: 245
  like: 206
  calling: 193
  um: 184
  don: 184
  want: 176
  hello: 171
  thank: 170


In [11]:
# look for specific scam keywords
scam_keywords = {
    'tech_support': ['computer', 'windows', 'microsoft', 'virus', 'error', 'refund'],
    'financial': ['bank', 'account', 'card', 'money', 'payment', 'transfer'],
    'authority': ['police', 'irs', 'social security', 'officer', 'warrant', 'legal'],
    'urgency': ['immediately', 'urgent', 'now', 'today', 'expires']
}

for category, keywords in scam_keywords.items():
    count = 0
    for keyword in keywords:
        count += df['Content'].str.lower().str.contains(keyword, na=False).sum()
    print(f"{category}: {count} mentions across transcripts")

tech_support: 255 mentions across transcripts
financial: 165 mentions across transcripts
authority: 66 mentions across transcripts
urgency: 250 mentions across transcripts


## enhanced model

## results

## what words predict scams

In [12]:
# examples of high urgency calls
high_urgency = df[df['urgency_score'] >= 3].sort_values('urgency_score', ascending=False)
print(f"calls with 3+ urgency words: {len(high_urgency)}\n")
if len(high_urgency) > 0:
    print("example (highest urgency):")
    print(high_urgency.iloc[0]['Content'][:300])

calls with 3+ urgency words: 0



In [13]:
# examples with verification requests
with_verification = df[df['verification_requests'] > 0]
print(f"calls asking for personal info: {len(with_verification)}\n")
if len(with_verification) > 0:
    print("example:")
    print(with_verification.iloc[0]['Content'][:300])

calls asking for personal info: 17

example:
hi mr how are you doing today fine i'm great who is this uh this is you know you have talked to me in previous time about your ppi compensation so what what do i need to do for this ppi uh compensation you know you have to complete your uh confirmation we need your sort code account number for verif


In [14]:
# summary stats
print("youtube scam calls dataset summary:")
print(f"total transcripts: {len(df)}")
print(f"avg length: {df['Char_Len'].mean():.0f} characters")
print(f"avg urgency score: {df['urgency_score'].mean():.2f}")
print(f"calls with urgency language: {(df['urgency_score'] > 0).sum()} ({(df['urgency_score'] > 0).sum()/len(df)*100:.1f}%)")
print(f"calls requesting verification: {(df['verification_requests'] > 0).sum()} ({(df['verification_requests'] > 0).sum()/len(df)*100:.1f}%)")

youtube scam calls dataset summary:
total transcripts: 243
avg length: 759 characters
avg urgency score: 1.03
calls with urgency language: 189 (77.8%)
calls requesting verification: 17 (7.0%)
